In [1]:
# =========================
# Required Libraries
# =========================
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

In [2]:
# -------------------------
# Part 1: Data Preprocessing
# -------------------------

# 1) Download required NLTK data (with safe guards)
def _ensure_nltk_resources():
    resources = ["punkt", "stopwords", "wordnet", "omw-1.4"]
    for r in resources:
        try:
            nltk.data.find(r if r != "punkt" else "tokenizers/punkt")
        except LookupError:
            nltk.download(r, quiet=True)

_ensure_nltk_resources()

STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

# 2) Preprocessing function
def preprocess_text(text: str, remove_stopwords: bool = True) -> str:
    """
    - lowercase
    - tokenize
    - remove non-alphabetic tokens
    - optional stopword removal
    - lemmatize
    - join back to a string
    Includes basic error handling.
    """
    if not isinstance(text, str):
        return ""

    try:
        text = text.lower()
        tokens = word_tokenize(text)

        cleaned = []
        for tok in tokens:
            if tok.isalpha():  # remove non-alphabetic
                if not remove_stopwords or tok not in STOPWORDS:
                    lemma = LEMMATIZER.lemmatize(tok)
                    cleaned.append(lemma)

        return " ".join(cleaned)
    except Exception:
        # Fallback: very defensive
        return ""

# 3) Load and prepare the dataset (four classes as required)
def load_dataset():
    categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']
    data = fetch_20newsgroups(
        subset="all",
        categories=categories,
        remove=("headers", "footers", "quotes"),
        shuffle=True,
        random_state=42,
    )
    df = pd.DataFrame({"text": data.data, "label": data.target})
    df.dropna(subset=["text"], inplace=True)
    label_map = {i: c for i, c in enumerate(data.target_names)}
    return df.reset_index(drop=True), label_map

# 4) Vectorize with TF-IDF and split
def vectorize_and_split(df, max_features=1500, test_size=0.2, random_state=42):
    df = df.copy()
    df["clean"] = df["text"].apply(preprocess_text)

    vectorizer = TfidfVectorizer(max_features=max_features)
    X = vectorizer.fit_transform(df["clean"])
    y = df["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    return X_train, X_test, y_train, y_test, vectorizer

In [3]:
# -------------------------
# Part 2: Model Training & Evaluation
# -------------------------

# Baseline models
def build_baseline_models():
    return {
        "LogisticRegression": LogisticRegression(max_iter=200, n_jobs=None),
        "MultinomialNB": MultinomialNB(),
        "LinearSVC": LinearSVC(),
    }

# Task 1: Add Decision Tree and Random Forest
def add_tree_models(models: dict):
    models = dict(models)  # copy
    # Tree-based models do not accept sparse input; we'll densify inside trainer only for these.
    models["DecisionTree"] = DecisionTreeClassifier(random_state=42)
    models["RandomForest"] = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    return models

# Evaluation framework (macro averages, and accuracy)
def evaluate(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def train_and_eval_all(X_train, X_test, y_train, y_test, include_trees=True):
    results = {}
    models = build_baseline_models()
    if include_trees:
        models = add_tree_models(models)

    for name, model in models.items():
        # Densify only for tree-based models
        if name in {"DecisionTree", "RandomForest"}:
            Xtr = X_train.toarray()
            Xte = X_test.toarray()
        else:
            Xtr, Xte = X_train, X_test

        model.fit(Xtr, y_train)
        pred = model.predict(Xte)
        results[name] = evaluate(y_test, pred)

    return results, models

In [4]:
# -------------------------
# Task 2: Update Score Comparison
# -------------------------

def compare_scores(results_dict: dict) -> pd.DataFrame:
    """
    Combine scores from all models, sort by accuracy,
    and present as a tidy DataFrame.
    """
    rows = []
    for model_name, metrics in results_dict.items():
        row = {"model": model_name}
        row.update(metrics)
        rows.append(row)
    df = pd.DataFrame(rows)
    df.sort_values(by="accuracy", ascending=False, inplace=True, ignore_index=True)
    return df

In [5]:
# -------------------------
# Task 3: Best Model Selection
# -------------------------

def select_best_by_f1(results_dict: dict, models: dict):
    best_name = max(results_dict, key=lambda k: results_dict[k]["f1"])
    return best_name, models[best_name]

def predict_on_texts(texts, vectorizer, model, label_map):
    # vectorize
    X = vectorizer.transform([preprocess_text(t) for t in texts])

    # densify if needed
    needs_dense = hasattr(model, "predict_proba") and not hasattr(X, "toarray") is False
    if isinstance(model, (DecisionTreeClassifier, RandomForestClassifier)):
        X = X.toarray()

    preds = model.predict(X)
    return [label_map[int(i)] for i in preds]

# =========================
# Main
# =========================
if __name__ == "__main__":
    print("Loading data...")
    df, label_map = load_dataset()
    print(f"Samples: {len(df)}")
    print("Label distribution:\n", df["label"].value_counts().sort_index().rename(index=label_map))

    print("\nVectorizing and splitting...")
    X_train, X_test, y_train, y_test, vectorizer = vectorize_and_split(df, max_features=1500)

    print("\nTraining & evaluating models...")
    results, trained_models = train_and_eval_all(X_train, X_test, y_train, y_test, include_trees=True)

    print("\nCombined Score Table (sorted by accuracy):")
    table = compare_scores(results)
    print(table.to_string(index=False))

    # Best model by F1
    best_name, best_model = select_best_by_f1(results, trained_models)
    print(f"\nBest model by F1-score: {best_name} (F1={results[best_name]['f1']:.4f})")

    # Test the best model on new texts
    test_texts = [
        "The new graphics card offers incredible rendering speed and 4K resolution support.",
        "The patient showed symptoms of high fever and respiratory issues.",
        "The discussion about faith and belief systems continued in the church meeting."
    ]
    preds = predict_on_texts(test_texts, vectorizer, best_model, label_map)

    print("\nPredictions on new texts:")
    for t, p in zip(test_texts, preds):
        print(f"- {p:>26s}  |  {t}")

Loading data...
Samples: 3759
Label distribution:
 label
alt.atheism               799
comp.graphics             973
sci.med                   990
soc.religion.christian    997
Name: count, dtype: int64

Vectorizing and splitting...

Training & evaluating models...

Combined Score Table (sorted by accuracy):
             model  accuracy  precision   recall       f1
LogisticRegression  0.841755   0.838993 0.833363 0.833998
     MultinomialNB  0.840426   0.848768 0.830199 0.832485
         LinearSVC  0.823138   0.819767 0.816240 0.817029
      RandomForest  0.799202   0.793383 0.786922 0.785232
      DecisionTree  0.703457   0.694462 0.693397 0.692310

Best model by F1-score: LogisticRegression (F1=0.8340)

Predictions on new texts:
-              comp.graphics  |  The new graphics card offers incredible rendering speed and 4K resolution support.
-                    sci.med  |  The patient showed symptoms of high fever and respiratory issues.
-     soc.religion.christian  |  The discuss